# ANIP - Tâche 1 : Reconnaissance Faciale Robuste
## Face Matching/Verification — Version Améliorée

**Améliorations par rapport à la version initiale :**
- Chemins relatifs (plus de chemins absolus macOS)
- Augmentation de données (flip, brightness, contrast)
- L2-normalisation des embeddings
- Fine-tuning des dernières couches de MobileNetV2
- Visualisation des courbes d'entraînement
- Courbe ROC + seuil optimal automatique
- Gestion d'erreurs sur les lectures d'images


## 1. Imports et Configuration

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
import cv2
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_v2_preprocess
try:
    from tensorflow.keras.applications import ConvNeXtTiny
except ImportError:
    ConvNeXtTiny = None
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Dense, Lambda, GlobalAveragePooling2D, Dropout, BatchNormalization
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, classification_report

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))


In [ ]:
# Chemins relatifs au notebook
NOTEBOOK_DIR = Path('.').resolve()
DATA_PATH = NOTEBOOK_DIR / 'anip-reconnaissance-faciale-estimation-ages-ocr' / 'dataset_tache_1' / 'dataset_tache_1'

# Fallback : chercher automatiquement
if not DATA_PATH.exists():
    candidates = list(NOTEBOOK_DIR.rglob('dataset_tache_1'))
    if candidates:
        DATA_PATH = candidates[-1]
        print(f'Dossier trouve : {DATA_PATH}')
    else:
        raise FileNotFoundError(f'Dossier dataset_tache_1 introuvable sous {NOTEBOOK_DIR}')

TRAIN_PATH = DATA_PATH / 'train'
TEST_PATH  = DATA_PATH / 'test'

IMG_SIZE   = (160, 160)
BATCH_SIZE = 32
EPOCHS     = 12
FINE_TUNE_EPOCHS = 6
SEED       = 42
VAL_SIZE   = 0.2

BACKBONE = 'mobilenetv2'  # 'mobilenetv2' ou 'convnext_tiny'
EMBED_DIM = 128
TRAIN_POS_PAIRS = 4000
TRAIN_NEG_PAIRS = 4000
VAL_POS_PAIRS   = 1000
VAL_NEG_PAIRS   = 1000

np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'Train : {TRAIN_PATH}')
print(f'Test  : {TEST_PATH}')
print(f'Backbone selectionne : {BACKBONE}')


## 2. Chargement et Analyse des Données

In [ ]:
def parse_train_filename(filepath):
    """Format: XXXX_Y.jpg -> XXXX=person_id, Y=photo_num"""
    parts = filepath.stem.split('_')
    return int(parts[0]), int(parts[1])


def load_train_data():
    """Charge les donnees d'entrainement avec validation des fichiers."""
    train_images = list(TRAIN_PATH.glob('*.jpg')) + list(TRAIN_PATH.glob('*.JPG'))
    data = []
    for img_path in train_images:
        try:
            person_id, photo_num = parse_train_filename(img_path)
            data.append({'filepath': str(img_path), 'person_id': person_id, 'photo_num': photo_num})
        except (ValueError, IndexError):
            print(f'  Fichier ignore (nom invalide) : {img_path.name}')
    return pd.DataFrame(data).sort_values(['person_id', 'photo_num']).reset_index(drop=True)


print('Chargement des donnees...')
df_train = load_train_data()
print(f"Images chargees   : {len(df_train)}")
print(f"Personnes uniques : {df_train['person_id'].nunique()}")
print(df_train.groupby('person_id').size().describe())

person_ids = np.array(sorted(df_train['person_id'].unique()))
train_ids, val_ids = train_test_split(person_ids, test_size=VAL_SIZE, random_state=SEED)

df_train_split = df_train[df_train['person_id'].isin(train_ids)].reset_index(drop=True)
df_val_split   = df_train[df_train['person_id'].isin(val_ids)].reset_index(drop=True)

print()
print('Split par identite :')
print(f'  Train identities : {len(train_ids)} | images : {len(df_train_split)}')
print(f'  Val identities   : {len(val_ids)} | images : {len(df_val_split)}')
print(f'  Overlap IDs      : {len(set(train_ids) & set(val_ids))}')


In [ ]:
# Distribution du nombre de photos par personne
photos_per_person = df_train.groupby('person_id').size()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(photos_per_person, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution : photos par personne')
axes[0].set_xlabel('Nombre de photos')
axes[0].set_ylabel('Nombre de personnes')
axes[0].grid(True, alpha=0.3)

# Afficher quelques images du dataset
sample = df_train.groupby('person_id').first().sample(8, random_state=SEED)
axes[1].axis('off')
plt.tight_layout()

fig2, axes2 = plt.subplots(2, 8, figsize=(18, 5))
sample_persons = df_train['person_id'].unique()[:8]
for col, pid in enumerate(sample_persons):
    person_imgs = df_train[df_train['person_id'] == pid]['filepath'].values
    for row, img_path in enumerate(person_imgs[:2]):
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (80, 80))
            axes2[row, col].imshow(img)
        axes2[row, col].axis('off')
        if row == 0:
            axes2[row, col].set_title(f'ID {pid}', fontsize=8)

fig2.suptitle('Exemples d\'images (2 photos par personne)', fontsize=12)
plt.tight_layout()
plt.show()


## 3. Création des Paires (Positive / Négative)

In [ ]:
def create_pairs(df, n_positive=2000, n_negative=2000, seed=SEED):
    """
    Cree des paires a partir d'un DataFrame contenant une seule partition
    (train ou validation), afin d'eviter toute fuite d'identite.
    """
    rng = np.random.default_rng(seed)
    pairs, labels = [], []
    seen = set()

    person_groups = {
        pid: grp['filepath'].values
        for pid, grp in df.groupby('person_id')
        if len(grp) >= 2
    }
    person_ids = np.array(list(person_groups.keys()))
    print(f'Personnes avec >= 2 photos : {len(person_ids)}')

    positive_candidates = []
    for pid, imgs in person_groups.items():
        for img_a, img_b in combinations(imgs, 2):
            positive_candidates.append((img_a, img_b, pid))

    rng.shuffle(positive_candidates)
    max_positive = len(positive_candidates) if n_positive is None else min(n_positive, len(positive_candidates))
    for img_a, img_b, pid in positive_candidates[:max_positive]:
        key = tuple(sorted((img_a, img_b)))
        seen.add(key)
        pairs.append([img_a, img_b])
        labels.append(1)

    if n_negative is None:
        n_negative = len(labels)

    attempts = 0
    negative_target = int(n_negative)
    while sum(1 for l in labels if l == 0) < negative_target and attempts < negative_target * 30:
        attempts += 1
        pid1, pid2 = rng.choice(person_ids, 2, replace=False)
        img1 = rng.choice(person_groups[pid1])
        img2 = rng.choice(person_groups[pid2])
        key = tuple(sorted((img1, img2)))
        if key not in seen:
            seen.add(key)
            pairs.append([img1, img2])
            labels.append(0)

    return np.array(pairs), np.array(labels)


pairs_train, labels_train = create_pairs(
    df_train_split,
    n_positive=TRAIN_POS_PAIRS,
    n_negative=TRAIN_NEG_PAIRS,
    seed=SEED,
)
pairs_val, labels_val = create_pairs(
    df_val_split,
    n_positive=VAL_POS_PAIRS,
    n_negative=VAL_NEG_PAIRS,
    seed=SEED + 1,
)

# Alias pour les cellules de visualisation existantes
pairs, labels = pairs_train, labels_train

print()
print(f'Train pairs : {len(pairs_train)}  (pos={labels_train.sum()}, neg={(labels_train == 0).sum()})')
print(f'Val pairs   : {len(pairs_val)}  (pos={labels_val.sum()}, neg={(labels_val == 0).sum()})')


In [ ]:
# Visualiser quelques paires positives et négatives
fig, axes = plt.subplots(2, 6, figsize=(16, 6))

pos_idx = np.where(labels == 1)[0][:3]
neg_idx = np.where(labels == 0)[0][:3]

for row_i, (indices, title_color) in enumerate([(pos_idx, 'green'), (neg_idx, 'red')]):
    for col_i, pair_idx in enumerate(indices):
        for k, img_path in enumerate(pairs[pair_idx]):
            ax = axes[row_i, col_i * 2 + k]
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (80, 80))
                ax.imshow(img)
            ax.axis('off')
        label = 'Même personne' if row_i == 0 else 'Différentes'
        axes[row_i, col_i * 2].set_title(f'{label} {col_i+1}', fontsize=8, color=title_color)

plt.suptitle('Exemples de paires positives (haut) et négatives (bas)', fontsize=12)
plt.tight_layout()
plt.show()


## 4. Générateur de Données avec Augmentation

In [ ]:
def get_backbone_preprocess(backbone_name):
    if backbone_name == 'mobilenetv2':
        return mobilenet_v2_preprocess
    if backbone_name == 'convnext_tiny':
        return lambda x: x
    raise ValueError(f'Backbone non supporte : {backbone_name}')


BACKBONE_PREPROCESS = get_backbone_preprocess(BACKBONE)


def load_and_preprocess_image(image_path, img_size=IMG_SIZE, augment=False):
    """Charge et pretraite une image. Retourne un tableau de zeros si illisible."""
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            raise ValueError(f'Impossible de lire : {image_path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, img_size)
        img = img.astype('float32')

        if augment:
            if np.random.rand() > 0.5:
                img = img[:, ::-1, :]
            delta = np.random.uniform(-18.0, 18.0)
            img = np.clip(img + delta, 0.0, 255.0)
            factor = np.random.uniform(0.9, 1.1)
            mean = img.mean(axis=(0, 1), keepdims=True)
            img = np.clip((img - mean) * factor + mean, 0.0, 255.0)

        img = BACKBONE_PREPROCESS(img)
        return img.astype('float32')
    except Exception as e:
        print(f'  [WARN] Image ignoree : {e}')
        return np.zeros((*img_size, 3), dtype='float32')


def pair_generator(pairs, labels, batch_size=32, shuffle=True, augment=False):
    """Generateur infini de paires d'images."""
    n = len(pairs)
    indices = np.arange(n)
    while True:
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]
            batch_pairs = pairs[batch_idx]
            batch_labels = labels[batch_idx]
            imgs1 = np.array([load_and_preprocess_image(p[0], augment=augment) for p in batch_pairs], dtype='float32')
            imgs2 = np.array([load_and_preprocess_image(p[1], augment=augment) for p in batch_pairs], dtype='float32')
            yield (imgs1, imgs2), batch_labels


## 5. Architecture du Réseau Siamois

In [ ]:
def build_backbone(backbone_name, input_shape):
    if backbone_name == 'mobilenetv2':
        model = MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape)
        model._name = 'mobilenetv2_backbone'
        fine_tune_last_n = 30
    elif backbone_name == 'convnext_tiny':
        if ConvNeXtTiny is None:
            raise ImportError(
                'ConvNeXtTiny n\'est pas disponible dans cette version de TensorFlow. '
                'Repasse BACKBONE a mobilenetv2.'
            )
        model = ConvNeXtTiny(include_top=False, weights='imagenet', input_shape=input_shape)
        model._name = 'convnext_tiny_backbone'
        fine_tune_last_n = 20
    else:
        raise ValueError(f'Backbone non supporte : {backbone_name}')
    return model, model.name, fine_tune_last_n


def create_base_network(input_shape):
    """Backbone ImageNet + tete d'embedding L2-normalisee."""
    backbone_model, backbone_name, fine_tune_last_n = build_backbone(BACKBONE, input_shape)
    backbone_model.trainable = False

    inputs = Input(shape=input_shape)
    x = backbone_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(EMBED_DIM, activation='linear')(x)
    embeddings = Lambda(lambda t: tf.math.l2_normalize(t, axis=1), name='l2_norm')(x)

    model = Model(inputs=inputs, outputs=embeddings, name='base_network')
    return model, backbone_name, fine_tune_last_n


def create_siamese_network(input_shape):
    """Reseau siamois avec distance L1 entre embeddings normalises."""
    base_network, backbone_name, fine_tune_last_n = create_base_network(input_shape)

    input_a = Input(shape=input_shape, name='input_a')
    input_b = Input(shape=input_shape, name='input_b')

    emb_a = base_network(input_a)
    emb_b = base_network(input_b)

    l1_dist = Lambda(lambda x: tf.abs(x[0] - x[1]), name='l1_distance')([emb_a, emb_b])
    output = Dense(1, activation='sigmoid', name='output')(l1_dist)

    siamese = Model(inputs=[input_a, input_b], outputs=output, name='siamese_network')
    return siamese, base_network, backbone_name, fine_tune_last_n


input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)
siamese_model, base_network, backbone_layer_name, fine_tune_last_n = create_siamese_network(input_shape)

siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)
siamese_model.summary()
print(f'Backbone actif : {backbone_layer_name} | fine-tune last n = {fine_tune_last_n}')


## 6. Entraînement — Phase 1 (Feature Extraction)

In [ ]:
steps_per_epoch = max(1, int(np.ceil(len(pairs_train) / BATCH_SIZE)))
validation_steps = max(1, int(np.ceil(len(pairs_val) / BATCH_SIZE)))

train_gen = pair_generator(pairs_train, labels_train, BATCH_SIZE, shuffle=True, augment=True)
val_gen = pair_generator(pairs_val, labels_val, BATCH_SIZE, shuffle=False, augment=False)

callbacks_phase1 = [
    keras.callbacks.ModelCheckpoint(
        'best_siamese_phase1.keras',
        monitor='val_auc', save_best_only=True, mode='max', verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=4, restore_best_weights=True, verbose=1, mode='max'
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1
    ),
]

print(f'Phase 1 : feature extraction ({BACKBONE} gele)...')
history1 = siamese_model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=EPOCHS,
    callbacks=callbacks_phase1,
    verbose=1,
)
print('Phase 1 terminee!')


## 7. Fine-Tuning — Phase 2 (Dernières Couches MobileNetV2)

In [ ]:
# Degeler uniquement les dernieres couches du backbone choisi
backbone_model = base_network.get_layer(backbone_layer_name)
backbone_model.trainable = True

if fine_tune_last_n > 0:
    for layer in backbone_model.layers[:-fine_tune_last_n]:
        layer.trainable = False

trainable_count = sum(1 for l in backbone_model.layers if l.trainable)
print(f'Couches entrainables dans {BACKBONE} : {trainable_count}')

siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)

train_gen2 = pair_generator(pairs_train, labels_train, BATCH_SIZE, shuffle=True, augment=True)
val_gen2 = pair_generator(pairs_val, labels_val, BATCH_SIZE, shuffle=False, augment=False)

callbacks_phase2 = [
    keras.callbacks.ModelCheckpoint(
        'best_siamese_final.keras',
        monitor='val_auc', save_best_only=True, mode='max', verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=4, restore_best_weights=True, verbose=1, mode='max'
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-8, verbose=1
    ),
]

print('Phase 2 : fine-tuning...')
history2 = siamese_model.fit(
    train_gen2,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen2,
    validation_steps=validation_steps,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=callbacks_phase2,
    verbose=1,
)
print('Fine-tuning termine!')


## 8. Courbes d'Entraînement

In [ ]:
def plot_history(h1, h2=None):
    acc      = h1.history['accuracy']     + (h2.history['accuracy']     if h2 else [])
    val_acc  = h1.history['val_accuracy'] + (h2.history['val_accuracy'] if h2 else [])
    loss     = h1.history['loss']         + (h2.history['loss']         if h2 else [])
    val_loss = h1.history['val_loss']     + (h2.history['val_loss']     if h2 else [])
    phase2_start = len(h1.history['accuracy']) if h2 else None

    epochs = range(1, len(acc) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for ax, train_v, val_v, title in [
        (ax1, acc,  val_acc,  'Accuracy'),
        (ax2, loss, val_loss, 'Loss'),
    ]:
        ax.plot(epochs, train_v, label='Train')
        ax.plot(epochs, val_v,   label='Validation')
        if phase2_start:
            ax.axvline(phase2_start, color='gray', linestyle='--', label='Fine-tuning')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle("Courbes d'entraînement — Réseau Siamois", fontsize=13)
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Courbes sauvegardées : training_curves.png')


plot_history(history1, history2)


## 9. Extraction d'Embeddings

In [ ]:
def get_embeddings(filepaths, base_network, batch_size=32):
    embeddings = []
    for i in tqdm(range(0, len(filepaths), batch_size), desc='Embeddings'):
        batch = filepaths[i:i + batch_size]
        imgs = np.array([load_and_preprocess_image(p) for p in batch], dtype='float32')
        embs = base_network.predict(imgs, verbose=0)
        embeddings.extend(embs)
    return np.array(embeddings)


val_embeddings = get_embeddings(df_val_split['filepath'].values, base_network)
df_val_embeddings = df_val_split.copy()
df_val_embeddings['embedding'] = list(val_embeddings)
print(f'Embeddings validation shape : {val_embeddings.shape}')


## 10. Calibration du Seuil par Courbe ROC

In [ ]:
# Dictionnaire filepath -> embedding sur le vrai split de validation
emb_dict = dict(zip(df_val_embeddings['filepath'].values, val_embeddings))

valid_mask = np.array([
    (p[0] in emb_dict) and (p[1] in emb_dict)
    for p in pairs_val
])
pairs_val_eval = pairs_val[valid_mask]
labels_val_eval = labels_val[valid_mask]

val_sims = np.array([
    float(np.dot(emb_dict[p[0]], emb_dict[p[1]]))
    for p in pairs_val_eval
])

fpr, tpr, thresholds = roc_curve(labels_val_eval, val_sims)
roc_auc = auc(fpr, tpr)

# Seuil optimal : indice de Youden (maximise TPR - FPR)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = float(thresholds[optimal_idx])

print(f'AUC-ROC       : {roc_auc:.4f}')
print(f'Seuil optimal : {optimal_threshold:.4f}  (TPR={tpr[optimal_idx]:.3f}, FPR={fpr[optimal_idx]:.3f})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
ax1.scatter(
    fpr[optimal_idx],
    tpr[optimal_idx],
    color='red',
    zorder=5,
    label=f'Seuil optimal = {optimal_threshold:.3f}',
)
ax1.plot([0, 1], [0, 1], 'k--')
ax1.set_xlabel('Taux de Faux Positifs')
ax1.set_ylabel('Taux de Vrais Positifs')
ax1.set_title('Courbe ROC')
ax1.legend()
ax1.grid(True, alpha=0.3)

bins = np.linspace(val_sims.min(), val_sims.max(), 50)
ax2.hist(val_sims[labels_val_eval == 1], bins=bins, alpha=0.6, label='Meme personne')
ax2.hist(val_sims[labels_val_eval == 0], bins=bins, alpha=0.6, label='Personnes diff.')
ax2.axvline(optimal_threshold, color='red', linestyle='--', label=f'Seuil = {optimal_threshold:.3f}')
ax2.set_xlabel('Similarite cosine')
ax2.set_ylabel('Nombre de paires')
ax2.set_title('Distribution des similarites')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

val_preds = (val_sims >= optimal_threshold).astype(int)
print('\nClassification Report (validation) :')
print(classification_report(labels_val_eval, val_preds, target_names=['Diff.', 'Meme'], zero_division=0))


## 11. Prédiction sur le Test Set

In [ ]:
def predict_test_set(test_path, base_network, threshold):
    test_images = list(test_path.glob('*.jpg')) + list(test_path.glob('*.JPG'))
    test_images = sorted(test_images)
    print(f'Images de test : {len(test_images)}')

    df_test = pd.DataFrame({
        'filepath': [str(p) for p in test_images],
        'filename': [p.name for p in test_images],
    })
    test_embs = get_embeddings(df_test['filepath'].values, base_network)
    df_test['embedding'] = list(test_embs)

    # Matrice de similarite vectorisee (dot product = cosine car embeddings L2-normalises)
    sim_matrix = test_embs @ test_embs.T

    matches = []
    n = len(df_test)
    for i in range(n):
        for j in range(i + 1, n):
            sim = float(sim_matrix[i, j])
            if sim >= threshold:
                matches.append({
                    'image1': df_test.iloc[i]['filename'],
                    'image2': df_test.iloc[j]['filename'],
                    'similarity': round(sim, 5),
                    'is_match': 1,
                })
    return pd.DataFrame(matches), df_test


matches_df, df_test = predict_test_set(TEST_PATH, base_network, threshold=optimal_threshold)
print(f'Paires matchees : {len(matches_df)}')
print(matches_df.head(10))


## 12. Fichier de Soumission

In [ ]:
matches_df.to_csv('tache1_submission.csv', index=False)
print('Soumission sauvegardee : tache1_submission.csv')
print(f'Seuil utilise : {optimal_threshold:.4f}')
print(f'AUC-ROC       : {roc_auc:.4f}')
print(f'Backbone      : {BACKBONE}')


## 13. Visualisation des Paires Matchées

In [ ]:
if len(matches_df) > 0:
    top_matches = matches_df.sort_values('similarity', ascending=False).head(3)
    fig, axes = plt.subplots(3, 2, figsize=(8, 12))

    for idx, (_, row) in enumerate(top_matches.iterrows()):
        for col, img_name in enumerate([row['image1'], row['image2']]):
            img = cv2.imread(str(TEST_PATH / img_name))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[idx, col].imshow(img)
            else:
                axes[idx, col].text(0.5, 0.5, 'Erreur', ha='center', va='center')
            label = img_name if col == 0 else f"{img_name}\nSim: {row['similarity']:.3f}"
            axes[idx, col].set_title(label, fontsize=9)
            axes[idx, col].axis('off')

    plt.suptitle('Top 3 Paires Matchées', fontsize=13)
    plt.tight_layout()
    plt.savefig('tache1_matches_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Visualisation sauvegardée : tache1_matches_visualization.png')
else:
    print('Aucune paire trouvée — essaie de baisser le seuil.')
